# Mapping Saturation

Calculate the saturation within the last 3 years.
Time interval is one month since 2008.
Different statistical models are used to find out if saturation is reached.

## Methods and Data
- Intrinsic approach
Premise: The added number of OSM objects of a specific feature class per time period decreases as the number of mapped objects converges against the (unknown) true number of objects.

Each aggregation of features (e.g. length of roads or count of building) has a maximum. After increased mapping activity saturation is reached near this maximum.

## Limitations
Informativeness varies across topics: performs better for small, discrete objects (e.g., houses) but worse for large polygons (e.g., land use), where a single abrupt mapping step can lead to low completeness.

## References

- Gröchenig S et al. (2014): Digging into the history of VGI data-sets: results from a worldwide study on OpenStreetMap mapping activity (https://doi.org/10.1080/17489725.2014.978403)
- Barrington-Leigh C and Millard-Ball A (2017): The world’s user-generated road map is more than 80% complete (https://doi.org/10.1371/journal.pone.0180698 pmid:28797037)
- Josephine Brückner, Moritz Schott, Alexander Zipf, and Sven Lautenbach (2021): Assessing shop completeness in OpenStreetMap for two federal states in Germany (https://doi.org/10.5194/agile-giss-2-20-2021)


In [1]:
import geojson
import requests
import geopandas as gpd
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys

indicator = "/mapping-saturation"
topic = "land-cover"
input_geom_path = "friedberg.geojson"
output_geom_path = "your_output_layer.gpkg"
max_workers = 20

base_url = "https://api.quality.ohsome.org/v1-test"
endpoint = "/indicators"
url = base_url + endpoint + indicator

gdf = gpd.read_file(input_geom_path)
gdf["result_value"] = pd.Series([None] * len(gdf), dtype="float")
gdf["response_time"] = pd.Series([None] * len(gdf), dtype="float")

headers = {"accept": "application/json"}

def fetch(index, geometry):
    bpolys = geojson.Feature(geometry=geometry)
    bpolys_collection = geojson.FeatureCollection([bpolys])

    parameters = {
        "topic": topic,
        "bpolys": bpolys_collection,
    }
    for attempt in range(5):
        try:
            print(f"posting request for index {index}")
            startresponse = time.time()
            response = requests.post(url, headers=headers, json=parameters, timeout=600)
            response.raise_for_status()
            result = response.json()
            endresponse = time.time()
            responsetime = endresponse - startresponse
            value = result["result"][0]["result"]["value"]
            return index, value, responsetime
        except requests.RequestException as e:
            print(f"Attempt {attempt + 1} failed at index {index}: {e}")
            if attempt < 3:
                print("Retrying...")
                time.sleep(2)
            else:
                print("Max retries reached. Skipping.")
                return index, None, None

start = time.time()
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(fetch, i, gdf.geometry.iloc[i]) for i in range(len(gdf))]

    for future in as_completed(futures):
        index, value, responsetime = future.result()
        gdf.at[index, "result_value"] = value
        gdf.at[index, "response_time"] = responsetime
        print(f"Completed index {index}: {value}")

end = time.time()
print(f"Calculation took {end - start:.2f} seconds")

gdf.to_file(output_geom_path, driver="GPKG")

posting request for index 0
Completed index 0: 0.9684271779448536
Calculation took 7.83 seconds
